# 4 · Run-to-run consistency of the LLM extraction (sensitivity analysis)

This notebook produces the results of the paper appendix **"Sensitivity analysis:
run-to-run consistency"**:

| Output in this notebook | Paper table / claim |
|---|---|
| Issues per judgment / refs per aligned issue, per run | Table `tab:sens_counts` |
| Run-to-run agreement on issue extraction | Table `tab:sens_issues` |
| Judgments with identical segmentation (30/50) | In-text |
| Run-to-run agreement on citation extraction | Table `tab:sens_citations` |
| Aligned issues with zero reference overlap (7/63) | In-text |
| Comparison of the three agreement regimes | Table `tab:sens_comparison` |

**Setup.** The extraction pipeline (DeepSeek, temperature 0, fixed seed) was run twice on
the same 50 test judgments, producing runs R1 (*original*, the run used in the main
validation) and R2 (*new*, the replication). A tax-law expert aligned, judgment by
judgment, (1) the legal issues found by each run and (2), within each pair of aligned
issues, the individual cited documents (a single `<item>` may bundle several references —
the `*_final` columns contain the manual per-document counts). The comparison uses the
**raw** LLM outputs, before the hallucination filter.

**Metrics.** Treating R1 and R2 as two annotators: for a system run S and ground-truth run
G, P(S|G) = |S∩G|/|S| and R(S|G) = |S∩G|/|G|; the two directions exchange P and R and share
the same F1. Counts are pooled over the corpus (micro), matching the paper's tables.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

DATA = Path('../data/sensitivity')

issues = pd.read_csv(DATA / 'issue_counts_runs.csv')
refs   = pd.read_csv(DATA / 'reference_counts_runs.csv')
with open(DATA / 'alignments_runs.json', encoding='utf-8') as fh:
    raw = json.load(fh)

print(f"judgments (issue-level rows): {len(issues)}")
print(f"aligned-issue rows (reference-level): {len(refs)}")
print(f"raw JSON judgments: {len(raw)}")

judgments (issue-level rows): 50
aligned-issue rows (reference-level): 63
raw JSON judgments: 50


## Data-quality checks

Before any statistic we validate that the annotation is internally coherent and that the
derived CSVs agree with the raw annotator state (`alignments_runs.json`).

One historical anomaly, corrected at source: for `Sentenza_Z31_3163_2023.xml` (issue pair
Q2–Q2) the annotation tool originally recorded an intersection of 7 references, which
exceeds the run-1 count of 5 (an impossible value, due to an annotation slip). The value
was corrected to 5; the CSV column `n_refs_intersect_manual` retains the original raw
value (7) for transparency, while `n_refs_intersect` holds the corrected value used by
all statistics. The defensive clipping step below would in any case enforce
x ≤ min(orig, new).

In [2]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})

# -- coverage ---------------------------------------------------------------
n_files = issues["filename"].nunique()
check("All 50 judgments present", n_files == 50, f"{n_files} unique filenames")
check("No duplicate judgments", issues["filename"].is_unique,
      f"{issues['filename'].duplicated().sum()} duplicates")
check("All judgments marked reviewed (revisionato==1)",
      bool((issues["revisionato"] == 1).all()),
      f"{int((issues['revisionato']==1).sum())}/{len(issues)} reviewed")

# -- issue-level coherence --------------------------------------------------
bad = issues[issues["n_issues_aligned"] >
             issues[["n_issues_original", "n_issues_new"]].min(axis=1)]
check("aligned issues <= min(original, new)", bad.empty,
      f"{len(bad)} violating rows")
check("issue counts non-negative",
      bool((issues[["n_issues_original","n_issues_new","n_issues_aligned"]] >= 0).all().all()))

# -- reference rows match the number of aligned issues ----------------------
ref_rows = refs.groupby("filename").size()
exp_rows = issues.set_index("filename")["n_issues_aligned"]
join = exp_rows.to_frame("aligned").join(ref_rows.rename("ref_rows")).fillna(0).astype(int)
mismatch = join[join["aligned"] != join["ref_rows"]]
check("#reference rows == #aligned issues (per judgment)", mismatch.empty,
      f"{len(mismatch)} mismatching judgments")

# -- reference-level coherence ----------------------------------------------
ro, rn, xr = refs["n_refs_original_final"], refs["n_refs_new_final"], refs["n_refs_intersect"]
viol = refs[xr > np.minimum(ro, rn)]
check("intersection <= min(original, new) refs", viol.empty,
      f"{len(viol)} violating rows")
check("reference counts non-negative",
      bool((refs[["n_refs_original_final","n_refs_new_final","n_refs_intersect"]] >= 0).all().all()))

# -- cross-validate CSV against raw JSON ------------------------------------
json_aligned = {f: len(v.get("issueLinks", [])) for f, v in raw.items()}
csv_aligned  = issues.set_index("filename")["n_issues_aligned"].to_dict()
aligned_ok = all(json_aligned.get(f) == csv_aligned.get(f) for f in csv_aligned)
check("JSON issueLinks count == CSV n_issues_aligned", aligned_ok)

# intersection in CSV must equal refManual 'x' in JSON (up to the known
# correction documented above)
def json_intersect(row):
    d = raw.get(row["filename"], {}).get("refManual", {})
    # key is "<orig_idx>-<new_idx>"; ids are Q1.. -> idx = id-1
    oi = int(str(row["orig_issue_id"]).lstrip("Q")) - 1
    ni = int(str(row["new_issue_id"]).lstrip("Q")) - 1
    cell = d.get(f"{oi}-{ni}", {})
    v = cell.get("x", "")
    return int(v) if str(v).strip() != "" else None
refs["_json_x"] = refs.apply(json_intersect, axis=1)
x_match = refs.dropna(subset=["_json_x"])
x_diff = x_match[x_match["_json_x"].astype(int) != x_match["n_refs_intersect"]]
check("JSON refManual intersection == CSV n_refs_intersect",
      len(x_diff) == 0,
      f"checked {len(x_match)} rows; {len(x_diff)} differ (see note above)")

pd.DataFrame(checks)[["check", "status", "detail"]]

,check,status,detail
0,All 50 judgments present,PASS,50 unique filenames
1,No duplicate judgments,PASS,0 duplicates
2,All judgments marked reviewed (revisionato==1),PASS,50/50 reviewed
3,"aligned issues <= min(original, new)",PASS,0 violating rows
4,issue counts non-negative,PASS,
5,#reference rows == #aligned issues (per judgment),PASS,0 mismatching judgments
6,"intersection <= min(original, new) refs",PASS,0 violating rows
7,reference counts non-negative,PASS,
8,JSON issueLinks count == CSV n_issues_aligned,PASS,
9,JSON refManual intersection == CSV n_refs_inte...,PASS,checked 63 rows; 0 differ (see note above)


In [3]:
# Any rows where the CSV intersection differs from the raw JSON state would be
# listed here (none expected after the correction described above).
print(x_diff[["filename", "orig_issue_id", "new_issue_id",
              "n_refs_original_final", "n_refs_new_final",
              "n_refs_intersect", "_json_x"]].to_string(index=False)
      if len(x_diff) else "No differences between JSON and CSV.")

No differences between JSON and CSV.


In [4]:
# Defensive cleaning: clip impossible intersections to min(orig,new) for the
# statistics, while keeping the raw value for transparency. (The CSV is already
# consistent, so nothing is clipped here.)
refs = refs.copy()
refs["x_raw"] = refs["n_refs_intersect"]
refs["x"] = np.minimum(refs["n_refs_intersect"],
                       np.minimum(refs["n_refs_original_final"], refs["n_refs_new_final"]))
n_clipped = int((refs["x"] != refs["x_raw"]).sum())
print(f"{n_clipped} intersection value(s) clipped to satisfy x <= min(orig, new).")

0 intersection value(s) clipped to satisfy x <= min(orig, new).


## Issue-level consistency — Tables `tab:sens_counts` (left), `tab:sens_issues`

In [5]:
def prf_micro(O, N, X):
    "O=|R1|, N=|R2|, X=|R1∩R2| pooled. Returns dict for both directions."
    return {
        "R1 | R2": dict(P=X/O, R=X/N, F1=2*X/(O+N)),
        "R2 | R1": dict(P=X/N, R=X/O, F1=2*X/(O+N)),
    }

iss = issues.rename(columns={
    "n_issues_original": "n_o", "n_issues_new": "n_n", "n_issues_aligned": "n_x"}).copy()

tot_o, tot_n, tot_x = iss["n_o"].sum(), iss["n_n"].sum(), iss["n_x"].sum()
identical = int(((iss["n_o"] == iss["n_n"]) & (iss["n_o"] == iss["n_x"])).sum())

print(f"Issues — R1: {tot_o} total ({tot_o/len(iss):.2f} per judgment), "
      f"R2: {tot_n} total ({tot_n/len(iss):.2f} per judgment)")
print(f"Aligned (shared) issues: {tot_x}")
print(f"Judgments segmented identically by the two runs: {identical}/{len(iss)} "
      f"({identical/len(iss):.0%})")
print()
print("Run-to-run agreement on issue extraction — Table tab:sens_issues:")
print(pd.DataFrame(prf_micro(tot_o, tot_n, tot_x)).T.mul(100).round(1)[["P", "R", "F1"]].to_string())

Issues — R1: 78 total (1.56 per judgment), R2: 83 total (1.66 per judgment)
Aligned (shared) issues: 63
Judgments segmented identically by the two runs: 30/50 (60%)

Run-to-run agreement on issue extraction — Table tab:sens_issues:
            P     R    F1
R1 | R2  80.8  75.9  78.3
R2 | R1  75.9  80.8  78.3


## Reference-level consistency (within aligned issues) — Tables `tab:sens_counts` (right), `tab:sens_citations`

In [6]:
rf = refs.copy()
rf["ro"] = rf["n_refs_original_final"]
rf["rn"] = rf["n_refs_new_final"]

TO, TN, TX = rf["ro"].sum(), rf["rn"].sum(), rf["x"].sum()
zero_overlap = int((rf["x"] == 0).sum())

print(f"References within the {len(rf)} aligned issues — "
      f"R1: {TO} total ({TO/len(rf):.2f} per issue), R2: {TN} total ({TN/len(rf):.2f} per issue)")
print(f"Shared references: {TX}")
print(f"Aligned issues with zero reference overlap: {zero_overlap}/{len(rf)} "
      f"({zero_overlap/len(rf):.0%})")
print()
print("Run-to-run agreement on citation extraction — Table tab:sens_citations:")
print(pd.DataFrame(prf_micro(TO, TN, TX)).T.mul(100).round(1)[["P", "R", "F1"]].to_string())

References within the 63 aligned issues — R1: 233 total (3.70 per issue), R2: 259 total (4.11 per issue)
Shared references: 180
Aligned issues with zero reference overlap: 7/63 (11%)

Run-to-run agreement on citation extraction — Table tab:sens_citations:
            P     R    F1
R1 | R2  77.3  69.5  73.2
R2 | R1  69.5  77.3  73.2


## Counts table — Table `tab:sens_counts`

In [7]:
counts = pd.DataFrame({
    'Issues per judgment (N=50)': [f"{tot_o/len(iss):.2f} ({tot_o})", f"{tot_n/len(iss):.2f} ({tot_n})"],
    f'Refs per aligned issue (N={len(rf)})': [f"{TO/len(rf):.2f} ({TO})", f"{TN/len(rf):.2f} ({TN})"],
}, index=['R1 (original)', 'R2 (replication)'])
print('Average number of items per unit (totals in parentheses) — Table tab:sens_counts:')
print(counts.to_string())

Average number of items per unit (totals in parentheses) — Table tab:sens_counts:
                 Issues per judgment (N=50) Refs per aligned issue (N=63)
R1 (original)                     1.56 (78)                    3.70 (233)
R2 (replication)                  1.66 (83)                    4.11 (259)


## Cross-check: count tables vs the shipped XML extractions

The repository ships the raw XML outputs of the two runs
(`data/xml/extraction_run1`, `data/xml/extraction_run2`). Here we verify that the
issue counts recorded in `issue_counts_runs.csv` (compiled with the manual alignment
tool) match the actual XML files.

In [8]:
import xml.etree.ElementTree as ET

def xml_issue_counts(folder):
    counts = {}
    for f in sorted((Path('../data/xml') / folder).glob('*.xml')):
        counts[f.name] = len(ET.parse(f).getroot().findall('questione'))
    return pd.Series(counts)

cnt_r1 = xml_issue_counts('extraction_run1')
cnt_r2 = xml_issue_counts('extraction_run2')
chk = issues.set_index('filename')[['n_issues_original', 'n_issues_new']].copy()
chk['xml_r1'] = cnt_r1
chk['xml_r2'] = cnt_r2
bad = chk[(chk['n_issues_original'] != chk['xml_r1']) | (chk['n_issues_new'] != chk['xml_r2'])]
print(f"Judgments checked: {len(chk)}; mismatches: {len(bad)}")
if len(bad):
    print(bad.to_string())

Judgments checked: 50; mismatches: 0


## Comparison of the three agreement regimes — Table `tab:sens_comparison`

The Human–Human and LLM–Human figures below are computed in notebooks
`01_issue_extraction.ipynb` and `02_citation_extraction.ipynb` (pooled F1 on the N=20
shared set, Tables `tab:issue_pairwise` and `tab:cit_pairwise`); they are repeated here
for the side-by-side view of the paper's Table `tab:sens_comparison`.

In [9]:
f1_issues_rr = 100 * 2 * tot_x / (tot_o + tot_n)
f1_cits_rr   = 100 * 2 * TX / (TO + TN)

comparison = pd.DataFrame({
    'Issue F1': {
        'Human–Human (IAA)':  '88.1',          # notebook 01, A1|A2 pooled (N=20)
        'LLM–Human (range)':  '72.7–83.6',     # notebook 01, LLM|A1 and LLM|A2 pooled (N=20)
        'Run–Run (R1 vs R2)': f'{f1_issues_rr:.1f}',
    },
    'Citation F1': {
        'Human–Human (IAA)':  '97.1',          # notebook 02, A1|A2 pooled (N=20)
        'LLM–Human (range)':  '73.8–74.8',     # notebook 02, LLM|A1 and LLM|A2 pooled (N=20)
        'Run–Run (R1 vs R2)': f'{f1_cits_rr:.1f}',
    },
})
print('Agreement (F1, %) under the three regimes — Table tab:sens_comparison:')
print(comparison.to_string())

Agreement (F1, %) under the three regimes — Table tab:sens_comparison:
                     Issue F1 Citation F1
Human–Human (IAA)        88.1        97.1
LLM–Human (range)   72.7–83.6   73.8–74.8
Run–Run (R1 vs R2)       78.3        73.2
